# Text against fields

The second half of the question in the README: does the text add anything to a model built on the complaint's other fields? Three models are scored on the same complaints: the fields from the complaint-escalation project, the text, and both together. Which form each model takes is settled with cross-validation on train. After that the test set is used, once. The notebook takes about 8 minutes to run.

In [1]:
import sys

sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from src.data import load_text, time_split
from src.fields import build_features, fields_logistic, fields_xgboost
from src.models import chosen, cross_validate, fold_scores

In [2]:
train, test = time_split(load_text())
X_train, X_test = train["text"], test["text"]
y_train, y_test = train["disputed"], test["disputed"]
opening = X_train.str.lower().str[:100]

## The fields

`build_features()` in src/fields.py is the one from the complaint-escalation project. Product, sub-product, company response, timely response, tags, month and weekday are one-hot encoded, and the 30 most common issues get a column each. Three columns are numbers: the company's share of the complaints, its dispute rate in train (cross-fitted, so a complaint's own label isn't in it) and the days between the complaint coming in and being sent to the company. Channel and consent are left out, since every complaint with a narrative came through the web with consent. Every encoder is fitted on train.

In [3]:
F_train, F_test = build_features(train, test)
F_train.shape, F_test.shape

((112256, 123), (37405, 123))

In [4]:
F_train[["company_share", "company_rate", "days_to_send"]].describe().round(3)

,company_share,company_rate,days_to_send
count,112256.000,112256.000,112256.000
mean,0.023,0.233,2.501
std,0.024,0.051,12.609
min,0.000,0.053,0.000
25%,0.001,0.194,0.000
50%,0.013,0.235,0.000
75%,0.046,0.274,1.000
max,0.072,0.637,838.000


123 columns for the 112,256 training and 37,405 test complaints, and 120 of them are one-hot. The company rates run from 5.3% to 63.7%. At least half of the complaints were sent to the company on the day they came in, and the slowest took 838 days.

## The text score

The text model's score goes into the combined model as one more column, `text_score`. For the training rows it has to be out-of-fold. A score from a model that has seen the complaint's own label looks better than it is, and the combined model would learn to trust the text too much. `cross_validate()` already returns out-of-fold scores, on the same grouped folds as in the earlier notebooks. The test rows get their score further down, from the text model fitted on all of train.

In [5]:
text_train, text_cv = cross_validate(chosen(), X_train, y_train, groups=opening)
B_train = F_train.assign(text_score=text_train)
text_cv.mean()

roc_auc          0.649935
avg_precision    0.364023
dtype: float64

0.650 ROC-AUC and 0.364 average precision, as in the models notebook. These out-of-fold scores are the `text_score` column of `B_train`.

## Cross-validation on train

Five models on the same grouped folds: the text alone, the fields alone, and the fields with the text score added. The fields are tried with logistic regression and with XGBoost at the settings chosen in the complaint-escalation project.

One caveat for the two models with the text score. The text scores were made on these same folds, so the text model that scored a training fold had seen the validation fold's labels. A validation complaint's own text score never saw its own label, so the effect should be small, and the test run doesn't have this problem.

In [6]:
models = {
    "fields, logistic regression": (fields_logistic(), F_train),
    "fields, XGBoost": (fields_xgboost(), F_train),
    "fields and text score, logistic regression": (fields_logistic(text_score=True), B_train),
    "fields and text score, XGBoost": (fields_xgboost(), B_train),
}

oof = {}
folds = {"text": text_cv}
for name, (model, X) in models.items():
    oof[name], folds[name] = cross_validate(model, X, y_train, groups=opening)

pd.DataFrame({name: scores.mean() for name, scores in folds.items()}).T.round(4)

,roc_auc,avg_precision
text,0.6499,0.3640
"fields, logistic regression",0.6263,0.3206
"fields, XGBoost",0.6277,0.3238
"fields and text score, logistic regression",0.6759,0.3857
"fields and text score, XGBoost",0.6727,0.3810


In [7]:
pd.DataFrame({name: scores["roc_auc"] for name, scores in folds.items()}).round(4)

,text,"fields, logistic regression","fields, XGBoost","fields and text score, logistic regression","fields and text score, XGBoost"
0,0.6525,0.6274,0.6302,0.6786,0.6759
1,0.6494,0.6289,0.6297,0.6761,0.6729
2,0.6514,0.6245,0.6260,0.6763,0.6716
3,0.6407,0.6263,0.6261,0.6693,0.6674
4,0.6557,0.6244,0.6268,0.6793,0.6756


On their own the fields get 0.626 with logistic regression and 0.628 with XGBoost. The first project's logistic regression got 0.646 in CV on all complaints. Two of its columns, channel and consent, are constant here, so the fields have less to work with. The text alone is ahead of both at 0.650.

With the text score added to the fields, logistic regression gets 0.676, which is 0.050 above the fields and 0.026 above the text, and XGBoost gets 0.673. The order is the same in all five folds: fields, then text, then both.

Logistic regression goes to the test run, for the fields and for the combined model. It's level with XGBoost on the fields and slightly ahead once the text score is in, and the text score gets a single weight that can be read.

## The company, in the fields and in the text

The explain notebook found company names among the text model's largest weights, and the fields have the company too, as `company_share` and `company_rate`. To see how much of the text's gain is the company, the two logistic regressions are run again without those two columns.

In [8]:
company = ["company_share", "company_rate"]
no_company = {
    "fields": (fields_logistic(company=False), F_train.drop(columns=company)),
    "fields and text score": (fields_logistic(company=False, text_score=True), B_train.drop(columns=company)),
}

rows = {}
for name, (model, X) in no_company.items():
    _, scores = cross_validate(model, X, y_train, groups=opening)
    rows[name] = {"without the company": scores["roc_auc"].mean(),
                  "with the company": folds[name + ", logistic regression"]["roc_auc"].mean()}
pd.DataFrame(rows).T.round(4)

,without the company,with the company
fields,0.6067,0.6263
fields and text score,0.6731,0.6759


In [9]:
pd.DataFrame({
    "text_score": text_train,
    "company_rate": F_train["company_rate"],
    "fields_score": oof["fields, logistic regression"],
}).corr(method="spearman").round(3)

,text_score,company_rate,fields_score
text_score,1.000,0.297,0.304
company_rate,0.297,1.000,0.681
fields_score,0.304,0.681,1.000


Without the company columns the fields drop from 0.626 to 0.607, so the company is worth 0.020 to the fields on their own. Once the text score is in, it's worth 0.003 (0.673 to 0.676). The text already carries most of what the company columns know.

Seen from the text's side, it adds 0.066 to fields that don't have the company and 0.050 to fields that do. So about a quarter of what the text adds is the company, and the rest is something the fields don't have. The rank correlation between the text score and the fields score is 0.30, low enough for the two to help each other.

## The test run

This is the one use of the test set: 37,405 complaints received from 1 Oct 2016 to 28 Feb 2017. All three models are fitted on the whole training set. The combined model is fitted on the out-of-fold text scores, and the test complaints get their text score from the text model fitted on all of train.

In [10]:
text_test = chosen().fit(X_train, y_train).predict_proba(X_test)[:, 1]
B_test = F_test.assign(text_score=text_test)

fields_model = fields_logistic().fit(F_train, y_train)
both_model = fields_logistic(text_score=True).fit(B_train, y_train)

test_scores = pd.DataFrame({
    "fields": fields_model.predict_proba(F_test)[:, 1],
    "text": text_test,
    "both": both_model.predict_proba(B_test)[:, 1],
})
pd.DataFrame({name: fold_scores(y_test, test_scores[name]) for name in test_scores}).T.round(4)

,roc_auc,avg_precision
fields,0.6260,0.2887
text,0.6419,0.3190
both,0.6709,0.3436


In [11]:
y_test.mean()

np.float64(0.20839459965245288)

On the test set the fields get a ROC-AUC of 0.626, the text 0.642 and the two together 0.671, the same order as in cross-validation. The fields score what they did in CV, the text is 0.008 lower (0.650 in CV) and the combined model 0.005 lower (0.676).

Average precision is lower for all three than in CV, 0.289, 0.319 and 0.344 against 0.321, 0.364 and 0.386. The dispute rate in the test window is 20.8% instead of 23.2%, and average precision moves with it. Measured against the base rate, the combined model is at 1.65 times on test and 1.66 in CV.

In [12]:
pattern = "scottrade|rush ?card"
events = X_train.str.contains(pattern, case=False).to_numpy()
print(events.sum(), X_test.str.contains(pattern, case=False).sum())

pd.Series({
    "text on train, out-of-fold": roc_auc_score(y_train, text_train),
    "without the Scottrade and RushCard narratives": roc_auc_score(y_train[~events], text_train[~events]),
}).round(4)

619 20


text on train, out-of-fold                       0.6499
without the Scottrade and RushCard narratives    0.6471
dtype: float64

The explain notebook flagged Scottrade and RushCard, two events from late 2015 with some of the text model's largest weights. 619 training narratives mention one of them and only 20 test narratives do. Without those 619 the text's out-of-fold ROC-AUC on train is 0.647 instead of 0.650. So about a third of the text's drop from CV to test is those two events, which were easy to rank in train and were over by the test months.

### Without the template letters

In [13]:
seen_opening = test["text"].str.lower().str[:100].isin(opening)
seen_text = test["text"].str.lower().isin(X_train.str.lower())
print(seen_opening.sum(), seen_text.sum())

pd.DataFrame({name: fold_scores(y_test[~seen_opening], test_scores.loc[~seen_opening, name])
              for name in test_scores}).T.round(4)

286 77


,roc_auc,avg_precision
fields,0.6263,0.2900
text,0.6407,0.3187
both,0.6698,0.3433


286 test narratives start with the same 100 characters as a training narrative and 77 of them are exact copies, the counts from the split notebook. Without the 286, the text and the combined model lose about 0.001 (0.641 and 0.670) and the fields don't move. The template letters aren't what the result rests on.

### Is the gain more than noise

The test complaints are resampled with replacement 1,000 times, and each time the three ROC-AUCs are computed on the same resample. The table has the middle 95% of each difference.

In [14]:
rng = np.random.default_rng(0)
y = y_test.to_numpy()
draws = []
for _ in range(1000):
    rows = rng.integers(0, len(y), len(y))
    auc = {name: roc_auc_score(y[rows], test_scores[name].to_numpy()[rows]) for name in test_scores}
    draws.append({
        "both - fields": auc["both"] - auc["fields"],
        "both - text": auc["both"] - auc["text"],
        "text - fields": auc["text"] - auc["fields"],
    })
pd.DataFrame(draws).quantile([0.025, 0.5, 0.975]).T.round(4)

,0.025,0.500,0.975
both - fields,0.0395,0.0446,0.0507
both - text,0.0251,0.0289,0.0326
text - fields,0.0079,0.0158,0.0243


The combined model is 0.045 above the fields, with a 95% interval of 0.040 to 0.051, and 0.029 above the text (0.025 to 0.033). Both intervals are well clear of zero, so the gain from putting the two together is more than sampling noise. The text alone is 0.016 above the fields alone, with a wider interval of 0.008 to 0.024.

### By product

In [15]:
by_product = {}
for product, size in test["product"].value_counts().items():
    if size < 2000:
        continue
    rows = (test["product"] == product).to_numpy()
    by_product[product] = {"narratives": size, "dispute_rate": y_test[rows].mean()}
    for name in test_scores:
        by_product[product][name] = roc_auc_score(y_test[rows], test_scores.loc[rows, name])
pd.DataFrame(by_product).T.astype({"narratives": int}).round(3)

,narratives,dispute_rate,fields,text,both
Debt collection,8227,0.197,0.647,0.649,0.686
Credit reporting,6950,0.205,0.634,0.656,0.686
Mortgage,6576,0.237,0.562,0.626,0.634
Student loan,4615,0.179,0.592,0.636,0.646
Credit card,4227,0.236,0.647,0.653,0.693
Bank account or service,3631,0.191,0.639,0.627,0.674
Consumer Loan,2195,0.230,0.604,0.592,0.626


The combined model is the best of the three in all seven products with at least 2,000 test narratives. The text helps most where the fields are weakest. In mortgages the fields get 0.562 and the combined model 0.634, in student loans 0.592 and 0.646, and in both products nearly all of that is the text (0.626 and 0.636 on its own).

In debt collection, credit cards and bank accounts the text and the fields are about level on their own, and together they gain about 0.04 over the better one. Bank accounts and consumer loans are the two products where the fields are ahead of the text.

## What the combined model leans on

Both logistic regressions scale their numeric columns, so those weights are per standard deviation. The one-hot columns are 0 or 1.

In [16]:
def weights(model):
    return pd.Series(model[-1].coef_[0], index=model[0].get_feature_names_out())


table = pd.DataFrame({"fields": weights(fields_model), "both": weights(both_model)})
table.loc[["text_score", "company_rate", "company_share", "days_to_send"]].round(3)

,fields,both
text_score,NaN,0.492
company_rate,0.257,0.137
company_share,0.013,-0.033
days_to_send,-0.070,-0.064


In [17]:
table.loc[table["fields"].abs().sort_values(ascending=False).index].head(12).round(3)

,fields,both
company_response_Untimely response,-3.143,-3.071
company_response_Closed,1.155,1.103
company_response_Closed with explanation,1.052,1.020
issue_Can't repay my loan,-0.465,-0.293
sub_product_Gift or merchant card,0.437,0.349
sub_product_Mortgage,0.433,0.146
timely_response_No,-0.411,-0.424
issue_Improper use of my credit report,0.370,0.198
company_response_Closed with non-monetary relief,0.356,0.321
sub_product_Title loan,0.343,0.441


One standard deviation in the text score moves the log-odds by 0.49, the largest weight among the numeric columns. The weight on `company_rate` falls from 0.26 to 0.14 once the text score is there, which is the overlap from the company section. Some product and issue weights shrink as well, such as `sub_product_Mortgage` from 0.43 to 0.15 and `issue_Communication tactics` from -0.33 to -0.14, since the narrative already says what the complaint is about.

The company's response barely moves. `Untimely response` stays near -3.1, and `Closed` and `Closed with explanation` stay near 1.1 and 1.0. Those are the three largest weights in both models. The consumer writes the narrative before the company answers, so the text can't know the response. That's the part of the fields the text has no way to replace.

## Scores for the decision step

In [18]:
keep = ["complaint_id", "date_received", "product", "company", "disputed"]
scored_train = train[keep].assign(
    fields=oof["fields, logistic regression"],
    text=text_train,
    both=oof["fields and text score, logistic regression"],
)
scored_test = pd.concat([test[keep], test_scores], axis=1)

scored_train.to_csv("../data/scored_train.csv", index=False)
scored_test.to_csv("../data/scored_test.csv", index=False)
scored_train.shape, scored_test.shape

((112256, 8), (37405, 8))

The scores of all three models are saved for the decision step: out-of-fold for the 112,256 training complaints and from the test run for the 37,405 test complaints. `data/` isn't in the repo, so this notebook has to be run before the next one.